## With library for splitting data, accuracy, confusion matrix

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix

dataset_path = "/content/drive/MyDrive/prison_dataset.csv"
dataset = pd.read_csv(dataset_path)

# Splitting dataset
X = dataset.drop(columns=["Recidivism - Return to Prison numeric"])
y = dataset["Recidivism - Return to Prison numeric"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)

    def _grow_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        num_classes = len(set(y))

        # Stopping criteria
        if depth == self.max_depth or num_classes == 1:
            return Node(value=self._most_common_label(y))

        # find best split
        best_feature, best_threshold = None, None
        best_gini = float('inf')

        for feature_idx in range(num_features):
            feature_values = X[:, feature_idx]
            thresholds = set(feature_values)

            for threshold in thresholds:
                left_indices = X[:, feature_idx] <= threshold
                right_indices = X[:, feature_idx] > threshold

                left_gini = self._gini_impurity(y[left_indices])
                right_gini = self._gini_impurity(y[right_indices])

                gini = (len(left_indices) / num_samples) * left_gini + \
                       (len(right_indices) / num_samples) * right_gini

                if gini < best_gini:
                    best_feature = feature_idx
                    best_threshold = threshold
                    best_gini = gini

        # create split
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold

        left_subtree = self._grow_tree(X[left_indices], y[left_indices], depth + 1)
        right_subtree = self._grow_tree(X[right_indices], y[right_indices], depth + 1)

        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _gini_impurity(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        gini = 1 - np.sum(probabilities ** 2)
        return gini

    def _most_common_label(self, y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._predict_value(x, self.root) for x in X])

    def _predict_value(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_value(x, node.left)
        else:
            return self._predict_value(x, node.right)

    def print_tree(self):
        self._print_node(self.root)

    def _print_node(self, node, indent=""):
        if node is None:
            return

        if node.value is not None:
            print(indent + "Leaf Node:", node.value)
            return

        print(indent + f"Feature: {X.columns[node.feature]}, Threshold: {node.threshold}")
        print(indent + "|-> Left:")
        self._print_node(node.left, indent + "   ")
        print(indent + "|-> Right:")
        self._print_node(node.right, indent + "   ")

# train decision tree
decision_tree = DecisionTree(max_depth=3)
decision_tree.fit(X_train.values, y_train.values)

# predict on test set
y_pred = decision_tree.predict(X_test.values)

# evaluation and results
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
conf_matrix = confusion_matrix(y_test, y_pred)
print("\n------------------------\nConfusion Matrix:")
print(conf_matrix)
print("\n------------------------\nTree structure:")
decision_tree.print_tree()

Accuracy: 0.7212317666126418

------------------------
Confusion Matrix:
[[1139  218]
 [ 642 1086]]

------------------------
Tree structure:
Feature: Fiscal Year Released, Threshold: 2010
|-> Left:
   Leaf Node: 1
|-> Right:
   Feature: Fiscal Year Released, Threshold: 2013
   |-> Left:
      Feature: Main Supervising District, Threshold: 3JD
      |-> Left:
         Leaf Node: 1
      |-> Right:
         Leaf Node: 1
   |-> Right:
      Feature: Release Type, Threshold: Discharged End of Sentence
      |-> Left:
         Leaf Node: 0
      |-> Right:
         Leaf Node: 0


## All from scratch

In [ ]:
def split_data(X, y, test_ratio=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    num_samples = len(X)
    num_test_samples = int(test_ratio * num_samples)
    test_indices = np.random.choice(num_samples, size=num_test_samples, replace=False)
    train_indices = np.setdiff1d(np.arange(num_samples), test_indices)

    X_train = X[train_indices]
    y_train = y[train_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test

def accuracy_score(y_true, y_pred):
    correct = 0
    total = len(y_true)
    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1
    return correct / total

def confusion_matrix(y_true, y_pred):
    num_classes = len(set(y_true))
    matrix = [[0] * num_classes for _ in range(num_classes)]
    for true, pred in zip(y_true, y_pred):
        matrix[true][pred] += 1
    return matrix

dataset_path = "/content/drive/MyDrive/prison_dataset.csv"
dataset = pd.read_csv(dataset_path)

# Splitting dataset
X = dataset.drop(columns=["Recidivism - Return to Prison numeric"])
y = dataset["Recidivism - Return to Prison numeric"]
X_train, X_test, y_train, y_test = split_data(X.values, y.values, test_ratio=0.2, random_state=42)

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)

    def _grow_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        num_classes = len(set(y))

        # Stopping criteria
        if depth == self.max_depth or num_classes == 1:
            return Node(value=self._most_common_label(y))

        # find best split
        best_feature, best_threshold = None, None
        best_gini = float('inf')

        for feature_idx in range(num_features):
            feature_values = X[:, feature_idx]
            thresholds = set(feature_values)

            for threshold in thresholds:
                left_indices = X[:, feature_idx] <= threshold
                right_indices = X[:, feature_idx] > threshold

                left_gini = self._gini_impurity(y[left_indices])
                right_gini = self._gini_impurity(y[right_indices])

                gini = (len(left_indices) / num_samples) * left_gini + \
                       (len(right_indices) / num_samples) * right_gini

                if gini < best_gini:
                    best_feature = feature_idx
                    best_threshold = threshold
                    best_gini = gini

        # create split
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold

        left_subtree = self._grow_tree(X[left_indices], y[left_indices], depth + 1)
        right_subtree = self._grow_tree(X[right_indices], y[right_indices], depth + 1)

        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _gini_impurity(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        gini = 1 - np.sum(probabilities ** 2)
        return gini

    def _most_common_label(self, y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._predict_value(x, self.root) for x in X])

    def _predict_value(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_value(x, node.left)
        else:
            return self._predict_value(x, node.right)

    def print_tree(self):
        self._print_node(self.root)

    def _print_node(self, node, indent=""):
        if node is None:
            return

        if node.value is not None:
            print(indent + "Leaf Node:", node.value)
            return

        print(indent + f"Feature: {X.columns[node.feature]}, Threshold: {node.threshold}")
        print(indent + "|-> Left:")
        self._print_node(node.left, indent + "   ")
        print(indent + "|-> Right:")
        self._print_node(node.right, indent + "   ")

# train decision tree
decision_tree = DecisionTree(max_depth=3)
decision_tree.fit(X_train, y_train)

# predict on test set
y_pred = decision_tree.predict(X_test)

# evaluation and results
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
conf_matrix = confusion_matrix(y_test, y_pred)
print("\n------------------------\nConfusion Matrix:")
print(conf_matrix)
print("\n------------------------\nTree structure:")
decision_tree.print_tree()

Accuracy: 0.7211413748378729

------------------------
Confusion Matrix:
[[1139, 218], [642, 1085]]

------------------------
Tree structure:
Feature: Fiscal Year Released, Threshold: 2010
|-> Left:
   Leaf Node: 1
|-> Right:
   Feature: Fiscal Year Released, Threshold: 2013
   |-> Left:
      Feature: Main Supervising District, Threshold: 3JD
      |-> Left:
         Leaf Node: 1
      |-> Right:
         Leaf Node: 1
   |-> Right:
      Feature: Release Type, Threshold: Discharged End of Sentence
      |-> Left:
         Leaf Node: 0
      |-> Right:
         Leaf Node: 0
